# tk-assistant · Эксперимент №2: 3 эпохи + GGUF-экспорт (одна сессия)

**Runtime → T4 GPU**, ячейки сверху вниз, сессию не закрывать до конца шага 4.

Шаг 2: обучение (~60 мин, 474 шага). Шаг 4: GGUF Q4_K_M (~20 мин).
Всё в одной сессии — промежуточный zip не нужен, адаптер сразу конвертируется в GGUF.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%capture
!pip install unsloth tensorboard

## 1. Свежий клон + чистое окружение

Проверь в выводе: последний коммит и `report_to: tensorboard`.

In [ ]:
%cd /content
!rm -rf tk-assistant
!git clone https://github.com/markkentos/tk-assistant.git
%cd /content/tk-assistant
!git log --oneline -1
!grep report_to configs/train_qlora_e3.yaml
import os
os.environ.pop("WANDB_API_KEY", None)
print("окружение почищено")

## 2. Обучение: 3 эпохи (~60 минут, 474 шага)

Следи за loss-строками. В конце — 10 пробных ответов: появились ли правильные
номера статей — главный признак успеха эксперимента.

In [ ]:
!python src/train_qlora.py --config configs/train_qlora_e3.yaml

## 3. Проверка адаптера + бэкап в Google Drive

При первом запуске Drive попросит доступ к диску — разреши.

In [ ]:
!ls -la outputs/adapter_e3_lr2e-4/
import os
sft = "outputs/adapter_e3_lr2e-4/adapter_model.safetensors"
assert os.path.exists(sft) and os.path.getsize(sft) > 1_000_000, \
    "адаптер не сохранился — обучение не завершилось"
print("Адаптер на месте:", round(os.path.getsize(sft) / 1e6, 1), "МБ")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/tk-assistant"
!cp -r outputs/adapter_e3_lr2e-4 "/content/drive/MyDrive/tk-assistant/"
!ls -la "/content/drive/MyDrive/tk-assistant/adapter_e3_lr2e-4/"
print("Бэкап в Drive готов")

## 4. GGUF Q4_K_M в той же сессии (~20 минут)

На выходе `*.gguf` (~4.4 ГБ) → скачивается в Загрузки → в `deploy/models/` проекта.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="outputs/adapter_e3_lr2e-4",
    max_seq_length=2048,
    load_in_4bit=True,
)
model.save_pretrained_gguf("gguf_out", tokenizer, quantization_method="q4_k_m")

import glob
gguf_files = glob.glob("gguf_out/**/*.gguf", recursive=True)
print("GGUF-файлы:", gguf_files)
assert gguf_files, "GGUF не создался"
GGUF_PATH = gguf_files[0]
print("Размер:", round(os.path.getsize(GGUF_PATH) / 1e9, 2), "ГБ")

## 5. Скачать GGUF (~4.4 ГБ → Загрузки → deploy/models/)

In [ ]:
from google.colab import files
files.download(GGUF_PATH)

## 6. Ответы на бенчмарке (той же сессией, ~15 минут)

Скачается `answers_tuned_e3.jsonl` — на случай если нужно перегенерировать.

In [ ]:
!python src/generate_answers.py --mode tuned --adapter outputs/adapter_e3_lr2e-4 --input eval/benchmark.jsonl --output eval/runs/answers_tuned_e3.jsonl
files.download("eval/runs/answers_tuned_e3.jsonl")